# Method 2 — Full pipeline & đánh giá (Kaggle T4)

Phase 5 của `docs/method2_plan.md`: chạy oracle + full pipeline, xuất predictions theo contract, đo latency 4 giai đoạn. Ngân sách ~1h GPU.

**Ba quy tắc sống còn trên Kaggle** (§8 `docs/method2_plan.md`):

1. Bật **Save & Run All (Commit)** cho job dài — session tương tác bị ngắt sau ~20 phút không tương tác, commit run chạy nền đủ 12h.
2. Checkpoint mỗi 500 step vào `/kaggle/working`, và **luôn** hỗ trợ `resume_from`.
3. Cache model HuggingFace thành Kaggle Dataset (`BAAI/bge-m3` ~2.3GB) thay vì tải lại mỗi session.


In [1]:
# ===== Cell 0: dò dataset + HF cache =====
# PHẢI chạy trước mọi import transformers: thư viện chốt cache lúc import,
# set HF_HOME sau đó thì không còn tác dụng.
import os
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')


def _dirs_within(base: Path, max_depth: int = 4):
    """Mọi thư mục tới độ sâu `max_depth`, bỏ qua `hub/` cho nhanh."""
    frontier, seen = [base], []
    for _ in range(max_depth):
        nxt = []
        for d in frontier:
            try:
                children = [c for c in d.iterdir() if c.is_dir() and c.name != 'hub']
            except (PermissionError, OSError):
                continue
            seen.extend(children)
            nxt.extend(children)
        frontier = nxt
    return seen


def find_root(marker: str, label: str) -> Path:
    """Tìm thư mục chứa `marker`.

    Kaggle mount theo dạng /kaggle/input/datasets/<user>/<ds>/<ds>/, và số tầng
    đổi theo cách upload. Dò theo marker thì không phải hardcode username hay
    độ sâu — upload kiểu nào cũng tìm ra.
    """
    for d in [INPUT_ROOT] + _dirs_within(INPUT_ROOT):
        if (d / marker).exists():
            return d
    raise SystemExit(
        f'Không tìm thấy {label}: không thư mục nào dưới {INPUT_ROOT} có {marker}.\n'
        'Kiểm tra đã Add đủ 3 dataset ở sidebar Input chưa.'
    )


SRC_ROOT = find_root('src/models/preflight.py', 'dataset src')
DATA_ROOT = find_root('method2/manifest.json', 'dataset data')
HF_HOME = find_root('hub/models--BAAI--bge-m3', 'dataset hf-cache')

print('SRC :', SRC_ROOT)
print('DATA:', DATA_ROOT)
print('HF  :', HF_HOME)

os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

# Có thư mục model chưa đủ — thiếu file trọng số thì lỗi chỉ lộ ra lúc nạp
# model, sau khi đã tốn thời gian cài đặt và copy.
for name in ('models--BAAI--bge-m3', 'models--xlm-roberta-base'):
    weights = [
        f for f in (HF_HOME / 'hub' / name).rglob('*')
        if f.is_file() and f.suffix in ('.safetensors', '.bin') and f.stat().st_size > 10**8
    ]
    assert weights, f'{name}: không có file trọng số > 100 MB'
    print(f'  {name}: {max(f.stat().st_size for f in weights) / 1024**3:.2f} GB')
print('\nHF cache OK')


SRC : /kaggle/input/datasets/dathq12/toolcalling-vi-src/toolcalling-vi-src
DATA: /kaggle/input/datasets/dathq12/toolcalling-vi-data/toolcalling-vi-data
HF  : /kaggle/input/datasets/dathq12/toolcalling-vi-hf-cache/toolcalling-vi-hf-cache
  models--BAAI--bge-m3: 2.12 GB
  models--xlm-roberta-base: 1.04 GB

HF cache OK


In [2]:
# ===== Cell 1: env — PIN version =====
# Ba package này quyết định API training VÀ tên metric của
# InformationRetrievalEvaluator. Đổi bản là đổi khoá metric, hỏng cả
# load_best_model_at_end lẫn khả năng so sánh giữa các run.
!pip install -q 'transformers==5.15.1' 'sentence-transformers==6.0.0' 'peft==0.20.0' \
                accelerate jsonschema rank_bm25 datasets

# PEFT 0.20 raise nếu image có torchao < 0.16. Method 2 không dùng
# torchao quantization nên gỡ hẳn là xong.
!pip uninstall -y -q torchao 2>/dev/null || true

# torch KHÔNG pin: Kaggle cài sẵn bản CUDA riêng, ép cài lại vừa chậm vừa
# dễ lệch CUDA runtime của image. Chỉ ghi nhận version vào manifest.
import torch

free, total = torch.cuda.mem_get_info()
n_gpu = torch.cuda.device_count()
print(torch.cuda.get_device_name(0), f'{free/1024**3:.1f} / {total/1024**3:.1f} GB free')
print('số GPU:', n_gpu)

# sentence-transformers tự bọc DataParallel khi thấy >1 GPU. Với GradCache
# gọi model hàng trăm lần mỗi step thì phí đồng bộ cộng dồn rất nhanh.
if n_gpu > 1:
    print('  >1 GPU — truyền --single-gpu cho MỌI lệnh train')

# T4 là Turing (sm_75), KHÔNG có bf16 phần cứng. torch vẫn có thể báo
# is_bf16_supported()=True vì hỗ trợ qua emulation, chậm hơn fp16.
# Giữ fp16 bất kể giá trị này.
print('bf16 (emulated trên T4, vẫn dùng fp16):', torch.cuda.is_bf16_supported())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 29.6 MB/s eta 0:00:00
Tesla T4 14.5 / 14.6 GB free
số GPU: 2
  >1 GPU — truyền --single-gpu cho MỌI lệnh train
bf16 (emulated trên T4, vẫn dùng fp16): True


In [3]:
# ===== Cell 2: copy code + data vào /kaggle/working =====
# Dataset chỉ đọc, mà code ghi checkpoint và dùng đường dẫn tương đối, nên
# phải copy sang thư mục ghi được. Dùng path đã dò ở Cell 0.
import shutil

WORK = Path('/kaggle/working')
# `scripts` cần thiết: benchmark_biencoder.py chạy trên Kaggle.
for name in ('src', 'configs', 'scripts'):
    target = WORK / name
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(SRC_ROOT / name, target)

# Dataset data bắt đầu thẳng bằng method2/ custom_vi/ benchmark_vi/ (KHÔNG có
# tầng `data/`), còn code tham chiếu `data/method2/...` → copy vào data/.
data_dir = WORK / 'data'
if data_dir.exists():
    shutil.rmtree(data_dir)
data_dir.mkdir(parents=True)
for child in DATA_ROOT.iterdir():
    dest = data_dir / child.name
    shutil.copytree(child, dest) if child.is_dir() else shutil.copy2(child, dest)

%cd /kaggle/working

import json, glob, sys
sys.path.insert(0, '/kaggle/working')
# HF_HOME đã set ở Cell 0, kế thừa sang mọi tiến trình con `!python`.

print('src    :', sorted(p.name for p in (WORK / 'src').iterdir()))
print('data   :', sorted(p.name for p in data_dir.iterdir()))


/kaggle/working
src    : ['README.md', 'data', 'evaluation', 'models']
data   : ['benchmark_vi', 'custom_vi', 'method2']


In [4]:
# ===== Cell 3: kiểm tra bản copy TRƯỚC khi preflight =====
# Preflight kiểm tra tính đúng đắn của dữ liệu; cell này kiểm tra bước copy —
# tách ra để khi hỏng thì biết ngay là hỏng ở đâu.
REQUIRED = [
    'data/method2/decontamination.json',
    'data/method2/manifest.json',
    'data/method2/tool_pool.json',
    'data/method2/biencoder/train.jsonl',
    'data/method2/biencoder/val.jsonl',
    'data/method2/biencoder/pairs_stats.json',
    'data/method2/crossencoder/train.jsonl',
    'data/method2/crossencoder/val.jsonl',
    'data/method2/label_stats.json',
    'data/custom_vi/v1/test_seen.jsonl',
    'data/benchmark_vi/test.jsonl',
    'configs/method2/biencoder.yaml',
    'configs/method2/crossencoder.yaml',
    'configs/method2/pinned_versions.json',
    'src/models/preflight.py',
]
missing = []
for rel in REQUIRED:
    path = WORK / rel
    if path.exists() and path.stat().st_size > 0:
        print(f'  {path.stat().st_size / 1024**2:8.2f} MB  {rel}')
    else:
        missing.append(rel)
        print(f'  {"THIẾU":>11}  {rel}')
assert not missing, f'Copy chưa đủ: {missing}'

# import được thì mới chắc src/ copy nguyên vẹn.
import importlib

importlib.import_module('src.models.preflight')
manifest = json.load(open('data/method2/manifest.json', encoding='utf-8'))
print('\nsnapshot commit:', manifest.get('git_commit'))
print('copy OK')


     10.81 MB  data/method2/decontamination.json
      0.00 MB  data/method2/manifest.json
      5.01 MB  data/method2/tool_pool.json
     46.20 MB  data/method2/biencoder/train.jsonl
      5.47 MB  data/method2/biencoder/val.jsonl
      0.00 MB  data/method2/biencoder/pairs_stats.json
     91.13 MB  data/method2/crossencoder/train.jsonl
     11.29 MB  data/method2/crossencoder/val.jsonl
      0.00 MB  data/method2/label_stats.json
      5.18 MB  data/custom_vi/v1/test_seen.jsonl
     14.18 MB  data/benchmark_vi/test.jsonl
      0.00 MB  configs/method2/biencoder.yaml
      0.00 MB  configs/method2/crossencoder.yaml
      0.00 MB  configs/method2/pinned_versions.json
      0.01 MB  src/models/preflight.py

snapshot commit: 6eb1b582b4eaba9ff8a9b51c4d88ae5bd0898465
copy OK


## Pre-flight — cổng fail-closed TRƯỚC mọi training

```
decontamination.json tồn tại
        ↓
SHA-256 == manifest.json
        ↓
overlap train/val/test == 0
        ↓
unseen positive leakage == 0
        ↓
package versions khớp bản đã pin
        ↓
CHO PHÉP TRAIN
```

Thiếu file hoặc hash lệch → job dừng ngay, **không rebuild tự động**. Nếu
experiment chính tự dựng lại index từ dữ liệu đang có trên máy thì ta mất
đúng thứ cần đảm bảo: bằng chứng model được train trên đúng split đã kiểm
định. Rebuild là lệnh preprocessing riêng, chạy ở local rồi upload lại:
`python -m src.models.sources decontaminate && python -m src.models.sources manifest`

Vì sao `val ∩ test` là rủi ro nặng nhất: dù không train trên query đó, việc
chọn checkpoint/hyperparameter bằng val vẫn khiến metric test lạc quan hơn
thực tế. `data/benchmark_vi` **giữ nguyên** — decontamination nằm ở tầng
dataset của Method 2 nên bốn method vẫn được đánh giá trên cùng một tập test.


In [5]:
# Exit code != 0 → dừng notebook, không chạy tiếp cell training nào.
!python -m src.models.preflight \
    --config configs/method2/biencoder.yaml \
    --require-gpu T4 \
    --output results/method2/preflight.json

preflight = json.load(open('results/method2/preflight.json', encoding='utf-8'))
assert preflight['passed'], f"Preflight KHÔNG ĐẠT: {preflight['failures']}"
print('preflight PASS —', len(preflight['checks']), 'check')


[PASS] commit SHA — 6eb1b582b4ea (từ manifest, không có .git)
[PASS] working tree sạch — không áp dụng — chạy từ snapshot
[PASS] config parse được — configs/method2/biencoder.yaml (de66078595eaff5a…)
[PASS] manifest tồn tại — data/method2/manifest.json
[PASS] decontamination.json tồn tại — data/method2/decontamination.json
[PASS] SHA-256 data/method2/decontamination.json — 37bf70a801f7f2cf…
[PASS] SHA-256 data/method2/tool_pool.json — 4b3357fa13adbd8c…
[PASS] SHA-256 data/method2/biencoder/train.jsonl — eaf94362480fe1d6…
[PASS] SHA-256 data/method2/crossencoder/train.jsonl — 24d0ef10e3724e01…
[PASS] overlap Bi-Encoder == 0 — {'test∩train': 0, 'test∩val': 0, 'train∩val': 0}
[PASS] overlap Cross-Encoder == 0 — {'test∩train': 0, 'test∩val': 0, 'train∩val': 0}
[PASS] unseen positive leakage == 0 — 0 tool
[PASS] hai stage dùng chung index — khớp
[PASS] version transformers — 5.15.1
[PASS] version sentence-transformers — 6.0.0
[PASS] version peft — 0.20.0
[PASS] version torch (ghi nhận) — 2.

In [6]:
# Số liệu split để đối chiếu bằng mắt trước khi tiêu giờ GPU.
stats = json.load(open('data/method2/biencoder/pairs_stats.json', encoding='utf-8'))
decon = stats['decontamination']

print('unique query/split :', stats['unique_queries_per_split'])
print('positive pairs     :', stats['n_positive_pairs'])
print('negative samples   :', stats['n_negative_samples'])
print('query trùng split  :', decon['n_overlapping_queries'], decon['overlapping_queries'])
print('sample bị loại     :', decon['rows_dropped_total'], decon['rows_dropped_by_transition'])
print('overlap còn lại    :', stats['split_overlap_after'])


unique query/split : {'test': 9331, 'train': 58829, 'val': 7819}
positive pairs     : 102100
negative samples   : 18334
query trùng split  : 1718 {'test∩train': 574, 'train∩val': 572, 'test∩train∩val': 542, 'test∩val': 30}
sample bị loại     : 32340 {'train->test': 26778, 'val->test': 3437, 'train->val': 2125}
overlap còn lại    : {'test∩train': 0, 'test∩val': 0, 'train∩val': 0}


# Phase 5 — Full pipeline & đánh giá

Bi-Encoder → Cross-Encoder → Validator, xuất `predictions.jsonl` theo
prediction contract rồi chấm bằng evaluator chung `src/evaluation`.

**Hai chế độ bắt buộc** (§6.3 `experimental_plan.md`):

| Chế độ | Nghĩa |
|---|---|
| `oracle` | đưa thẳng tool đúng vào Cross-Encoder → đo riêng extraction |
| `pipeline` | Bi-Encoder retrieve rồi Cross-Encoder extract → đo cả chuỗi |

Chênh lệch giữa hai chế độ cho biết lỗi nằm ở retrieval hay extraction.

**Ngưỡng τ đã FREEZE từ val** (`run02/thresholds.json`). Đây là lần chạy
trên **test** — tune lại ngưỡng ở đây là làm hỏng toàn bộ kết quả.

Cần 5 dataset: `src`, `data`, `hf-cache`, **`biencoder-run02`**,
**`crossencoder-run01`** (mỗi checkpoint là thư mục `final/`).


## Nạp hai checkpoint từ Kaggle Dataset

`/kaggle/working` không sống qua session nên cả hai model phải được
upload lại. Dò theo marker để không hardcode tên dataset — Bi-Encoder có
`modules.json` (sentence-transformers), Cross-Encoder có
`crossencoder_heads.pt`.


In [7]:
import re, shutil, subprocess, sys

BI = '/kaggle/working/artifacts/method2/biencoder/run02'
CE = '/kaggle/working/artifacts/method2/crossencoder/run01'


def find_optional(markers, max_depth=6):
    """Thư mục chứa TẤT CẢ marker. Nhiều marker là cố ý:

    `final/modules.json` khớp cả run01 lẫn run02 của Bi-Encoder — hai thư mục
    giống hệt nhau về cấu trúc. Nếu dataset run01 cũ còn được Add vào notebook
    này, nó có thể được chọn trước và toàn bộ đánh giá cuối chạy bằng
    checkpoint sai mà không có dấu hiệu gì. `thresholds.json` chỉ đi kèm run02
    (hiệu chỉnh sau Round 2) nên nó phân biệt được hai bên.
    """
    if isinstance(markers, str):
        markers = [markers]
    for d in [INPUT_ROOT] + _dirs_within(INPUT_ROOT, max_depth):
        if all((d / m).exists() for m in markers):
            return d
    return None


for dest, markers, label in (
    (BI, ['final/modules.json', 'thresholds.json'], 'Bi-Encoder run02'),
    (CE, ['final/crossencoder_heads.pt'], 'Cross-Encoder run01'),
):
    if Path(f'{dest}/final').exists():
        print(f'{label}: đã có sẵn')
        continue
    found = find_optional(markers)
    assert found, (
        f'THIẾU dataset {label}: không thư mục nào có đủ {markers}. '
        'Với Bi-Encoder, thresholds.json PHẢI nằm cùng cấp với final/ — '
        'đó cũng là thứ phân biệt run02 với run01.'
    )
    shutil.copytree(found / 'final', f'{dest}/final', dirs_exist_ok=True)
    print(f'{label}: nạp từ {found}')
    if 'thresholds.json' in markers:
        shutil.copy2(found / 'thresholds.json', f'{dest}/thresholds.json')

# Ngưỡng hiệu chỉnh trên val rồi FREEZE. Tự sửa nếu `strategy` lệch với
# `winner` đã tính sẵn trong chính file — bug từng gặp: config cũ hardcode
# strategy="absolute" nên file luôn ghi "absolute" dù calibration tự chọn
# "gap" tốt hơn. gap_delta đã tính đúng sẵn nên KHÔNG cần GPU để sửa — chỉ
# sửa bản copy trong /kaggle/working, không đụng tới Kaggle Dataset gốc.
from src.models.biencoder.evaluate import reconcile_strategy

raw_thr = json.load(open(f'{BI}/thresholds.json', encoding='utf-8'))
thr, _fixed = reconcile_strategy(raw_thr)
if _fixed:
    print(f'SỬA strategy: {raw_thr["strategy"]!r} -> {thr["strategy"]!r}',
          '(khớp winner đã tính sẵn trong calibration, không cần GPU)')
    json.dump(thr, open(f'{BI}/thresholds.json', 'w', encoding='utf-8'),
              ensure_ascii=False, indent=2)
print(f"tau={thr['tau']} tau_call={thr['tau_call']} gap_delta={thr['gap_delta']}",
      f"strategy={thr['strategy']} k_max={thr['k_max']}",
      f"hiệu chỉnh trên {thr['calibrated_on']}")


Bi-Encoder run02: nạp từ /kaggle/input/datasets/dathq12/toolcalling-vi-biencoder-run02
Cross-Encoder run01: nạp từ /kaggle/input/datasets/dathq12/toolcalling-vi-crossencoder-run01
SỬA strategy: 'absolute' -> 'gap' (khớp winner đã tính sẵn trong calibration, không cần GPU)
tau=0.35 tau_call=0.34 gap_delta=0.2 strategy=gap k_max=3 hiệu chỉnh trên data/method2/biencoder/val.jsonl


## Dựng lại index tool

`data/method2/index/` được sinh trong session Bi-Encoder và **không** nằm
trong dataset upload. Dựng lại mất ~50 s cho 4,464 tool — rẻ hơn nhiều so
với upload 18 MB embedding, và bảo đảm index khớp đúng checkpoint đang dùng.

`t_index_build` ghi riêng vào `index_meta.json`, **không** cộng vào latency
mỗi query (§10.1 `experimental_plan.md`) — đây là chi phí một lần.


In [8]:
if not Path('data/method2/index/tool_embeddings.npy').exists():
    subprocess.run(
        [sys.executable, '-m', 'src.models.biencoder.index',
         '--config', 'configs/method2/biencoder.yaml', '--model', f'{BI}/final'],
        check=True,
    )
else:
    print('index đã có — bỏ qua')

meta = json.load(open('data/method2/index/index_meta.json', encoding='utf-8'))
print('index:', meta['n_tools'], 'tool,', meta['t_index_build_sec'], 's')


Batches: 100%|██████████| 70/70 [00:46<00:00,  1.52it/s]


[index] 4464 tool × 1024d trong 46.172s → data/method2/index/tool_embeddings.npy
index: 4464 tool, 46.172 s


## Chạy pipeline + oracle trên 3 tập test

`custom_seen` · `custom_unseen` · `benchmark`, mỗi tập 2 chế độ = 6 lần
chạy. Ước tính ~1 h.


In [9]:
# subprocess thay vì `!` trong vòng lặp: exit code hiện ra rõ ràng nên một
# lần chạy hỏng không bị trôi qua trong Save & Run All.
GOLD = [
    ('data/custom_vi/v1/test_seen.jsonl', 'custom_seen'),
    ('data/custom_vi/v1/test_unseen.jsonl', 'custom_unseen'),
    ('data/benchmark_vi/test.jsonl', 'benchmark'),
]

for gold, tag in GOLD:
    for mode in ['pipeline', 'oracle']:
        out = f'results/method2/predictions/{tag}'
        done = f'{out}/' + ('predictions.jsonl' if mode == 'pipeline'
                            else 'oracle_predictions.jsonl')
        if Path(done).exists():
            print(f'=== {tag} / {mode}: đã có, bỏ qua ===', flush=True)
            continue
        print(f'=== {tag} / {mode} ===', flush=True)
        subprocess.run(
            [sys.executable, '-m', 'src.models.pipeline.method2',
             '--config', 'configs/method2/pipeline.yaml',
             '--gold', gold, '--mode', mode, '--output-dir', out],
            check=True,
        )


=== custom_seen / pipeline ===


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2714.06it/s]


{
  "n_samples": 800,
  "n_errors": 446,
  "latency": {
    "n": 800,
    "t_query_embed": {
      "mean_ms": 45.831,
      "p50_ms": 44.941,
      "p95_ms": 48.719,
      "p99_ms": 60.98
    },
    "t_retrieve": {
      "mean_ms": 0.086,
      "p50_ms": 0.081,
      "p95_ms": 0.116,
      "p99_ms": 0.138
    },
    "t_cross_encode": {
      "mean_ms": 14.347,
      "p50_ms": 13.742,
      "p95_ms": 41.253,
      "p99_ms": 65.437
    },
    "t_validate": {
      "mean_ms": 1.439,
      "p50_ms": 0.12,
      "p95_ms": 0.286,
      "p99_ms": 0.387
    },
    "total": {
      "mean_ms": 61.704,
      "p50_ms": 58.679,
      "p95_ms": 89.507,
      "p99_ms": 115.409
    },
    "gpu_peak_memory_mb": 3280.1
  },
  "output_dir": "results/method2/predictions/custom_seen"
}
=== custom_seen / oracle ===


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2513.87it/s]


{
  "n_samples": 800,
  "n_errors": 198,
  "latency": {
    "n": 800,
    "t_query_embed": {
      "mean_ms": 0.0,
      "p50_ms": 0.0,
      "p95_ms": 0.0,
      "p99_ms": 0.0
    },
    "t_retrieve": {
      "mean_ms": 0.0,
      "p50_ms": 0.0,
      "p95_ms": 0.0,
      "p99_ms": 0.0
    },
    "t_cross_encode": {
      "mean_ms": 11.349,
      "p50_ms": 10.545,
      "p95_ms": 36.974,
      "p99_ms": 54.564
    },
    "t_validate": {
      "mean_ms": 1.386,
      "p50_ms": 0.08,
      "p95_ms": 0.214,
      "p99_ms": 0.284
    },
    "total": {
      "mean_ms": 12.735,
      "p50_ms": 10.665,
      "p95_ms": 37.203,
      "p99_ms": 54.834
    },
    "gpu_peak_memory_mb": 3280.1
  },
  "output_dir": "results/method2/predictions/custom_seen"
}
=== custom_unseen / pipeline ===


...

{
  "n_samples": 800,
  "n_errors": 578,
  "latency": {
    "n": 800,
    "t_query_embed": {
      "mean_ms": 0.0,
      "p50_ms": 0.0,
      "p95_ms": 0.0,
      "p99_ms": 0.0
    },
    "t_retrieve": {
      "mean_ms": 0.0,
      "p50_ms": 0.0,
      "p95_ms": 0.0,
      "p99_ms": 0.0
    },
    "t_cross_encode": {
      "mean_ms": 10.548,
      "p50_ms": 12.223,
      "p95_ms": 38.094,
      "p99_ms": 45.242
    },
    "t_validate": {
      "mean_ms": 1.387,
      "p50_ms": 0.077,
      "p95_ms": 0.206,
      "p99_ms": 0.259
    },
    "total": {
      "mean_ms": 11.935,
      "p50_ms": 12.31,
      "p95_ms": 38.294,
      "p99_ms": 45.501
    },
    "gpu_peak_memory_mb": 3279.3
  },
  "output_dir": "results/method2/predictions/custom_unseen"
}
=== benchmark / pipeline ===


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2687.57it/s]


{
  "n_samples": 10555,
  "n_errors": 10395,
  "latency": {
    "n": 10555,
    "t_query_embed": {
      "mean_ms": 44.677,
      "p50_ms": 44.244,
      "p95_ms": 47.806,
      "p99_ms": 52.383
    },
    "t_retrieve": {
      "mean_ms": 0.058,
      "p50_ms": 0.055,
      "p95_ms": 0.087,
      "p99_ms": 0.108
    },
    "t_cross_encode": {
      "mean_ms": 14.432,
      "p50_ms": 11.981,
      "p95_ms": 33.473,
      "p99_ms": 64.229
    },
    "t_validate": {
      "mean_ms": 0.253,
      "p50_ms": 0.105,
      "p95_ms": 0.229,
      "p99_ms": 0.306
    },
    "total": {
      "mean_ms": 59.42,
      "p50_ms": 56.304,
      "p95_ms": 79.595,
      "p99_ms": 110.493
    },
    "gpu_peak_memory_mb": 3346.3
  },
  "output_dir": "results/method2/predictions/benchmark"
}
=== benchmark / oracle ===


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2464.83it/s]


{
  "n_samples": 10555,
  "n_errors": 9952,
  "latency": {
    "n": 10555,
    "t_query_embed": {
      "mean_ms": 0.0,
      "p50_ms": 0.0,
      "p95_ms": 0.0,
      "p99_ms": 0.0
    },
    "t_retrieve": {
      "mean_ms": 0.0,
      "p50_ms": 0.0,
      "p95_ms": 0.0,
      "p99_ms": 0.0
    },
    "t_cross_encode": {
      "mean_ms": 15.532,
      "p50_ms": 11.861,
      "p95_ms": 37.624,
      "p99_ms": 71.29
    },
    "t_validate": {
      "mean_ms": 0.243,
      "p50_ms": 0.092,
      "p95_ms": 0.205,
      "p99_ms": 0.291
    },
    "total": {
      "mean_ms": 15.776,
      "p50_ms": 11.955,
      "p95_ms": 37.844,
      "p99_ms": 71.439
    },
    "gpu_peak_memory_mb": 3346.3
  },
  "output_dir": "results/method2/predictions/benchmark"
}


## Evaluator chung — cả 3 tập

`--slice metadata.tool_split` tách seen/unseen, đúng như Bảng D §11
`experimental_plan.md` yêu cầu. `--oracle-predictions` cho phép evaluator
tính chênh lệch pipeline vs oracle trong cùng một báo cáo.


In [10]:
for gold, tag in GOLD:
    print(f'=== evaluate {tag} ===', flush=True)
    subprocess.run(
        [sys.executable, '-m', 'src.evaluation.cli', 'evaluate',
         '--gold', gold,
         '--predictions', f'results/method2/predictions/{tag}/predictions.jsonl',
         '--oracle-predictions',
         f'results/method2/predictions/{tag}/oracle_predictions.jsonl',
         '--slice', 'metadata.tool_split',
         '--output-dir', f'results/evaluation/method_2_{tag}'],
        check=True,
    )


=== evaluate custom_seen ===
=== evaluate custom_unseen ===
=== evaluate benchmark ===


## Chan doan: chien luoc chon tool

`raw_predictions_pipeline.jsonl` da luu `ranked_tools` kem score, nen doi
nguong hay doi chien luoc la **tinh lai duoc offline** - khong can GPU,
khong can chay lai pipeline.

Cot `#tool chon TB` la thu can nhin: CustomTools-VI co hard negative cung
`feature_group` nen diem sat nhau. Neu `absolute` chon trung binh 2-3 tool
trong khi gold chi 1, do chinh la cho Tool Set Accuracy roi.

Nguong chinh thuc VAN phai freeze tu val (§5.1) - bang nay chi de chan doan,
khong duoc chon nguong bang cach quet tren test.


In [11]:
for gold, tag in GOLD:
    raw_path = f'results/method2/predictions/{tag}/raw_predictions_pipeline.jsonl'
    if not Path(raw_path).exists():
        print(f'{tag}: chua co {raw_path}')
        continue
    print(f'=== {tag} ===')
    subprocess.run(
        [sys.executable, 'scripts/method2/replay_selection.py',
         '--gold', gold, '--raw', raw_path,
         '--thresholds', f'{BI}/thresholds.json', '--sweep'],
        check=True,
    )
    print()


=== custom_seen ===
gold 800 sample · raw 800 sample
thresholds: strategy='gap' winner-đã-tính='gap'

cấu hình                     toolset acc  #tool chọn TB   neg recall
--------------------------------------------------------------------
absolute                        0.6500           1.54       0.7450
gap  <- đang dùng               0.8550           1.29       0.7450
gap delta=0.05                  0.9450           1.09       0.7450
gap delta=0.1                   0.9825           1.13       0.7450
gap delta=0.15                  0.9900           1.16       0.7450
gap delta=0.25                  0.7675           1.38       0.7450
gap delta=0.3                   0.6800           1.52       0.7450
gap k_max=1                     0.8500           1.00       0.7450
gap k_max=2                     0.8750           1.27       0.7450

Ngưỡng chính thức vẫn phải freeze từ val — bảng này chỉ để chẩn đoán.

=== custom_unseen ===
gold 800 sample · raw 800 sample
thresholds: strategy='gap' win

## Bảng kết quả — Bảng C và D §11

Evaluator ghi `report.json`, `per_sample.jsonl`, `summary.md` vào
`results/evaluation/method_2_<tag>/`. Cell dưới đọc ra ba thứ:

1. **Bảng tổng** — metric chính của cả 3 tập, cạnh nhau.
2. **Chênh lệch oracle vs pipeline** — `oracle_arg_em − pipeline_arg_em`.
   Chênh nhỏ nghĩa là lỗi nằm ở extraction; chênh lớn nghĩa là retrieval
   đang kéo cả chuỗi xuống (§6.3 `experimental_plan.md`).
3. **Theo `tool_split`** — seen vs unseen, đúng Bảng D.

Cell này chỉ đọc file, không tốn GPU.


In [12]:
REPORTS = {}
for _, tag in GOLD:
    path = Path(f'results/evaluation/method_2_{tag}/report.json')
    if path.exists():
        REPORTS[tag] = json.load(open(path, encoding='utf-8'))
    else:
        print(f'THIẾU {path}')

ROWS = [
    ('Call F1',              lambda m: m['detection']['f1']),
    ('Recall@5',             lambda m: m['retrieval'].get('recall_at_5')),
    ('Tool Set Accuracy',    lambda m: m['selection']['tool_set_accuracy_positive']),
    ('ArgEM | đúng tool',    lambda m: m['extraction']['normalized_arg_em_given_correct_tool']),
    ('ArgEM | ORACLE tool',  lambda m: (m['oracle_extraction'] or {}).get('normalized_arg_em_given_correct_tool')),
    ('Argument Pair F1',     lambda m: m['extraction']['argument_pair']['f1']),
    ('Schema Validity',      lambda m: m['schema_validity']['call_schema_validity']),
    ('N-FCEM positive',      lambda m: m['end_to_end']['n_fcem_positive']),
    ('Overall Success',      lambda m: m['end_to_end']['overall_success']),
]


def fmt(v):
    return '  —   ' if v is None else f'{v:6.4f}'


tags = list(REPORTS)
print(f"{'metric':22s}" + ''.join(f'{t:>16s}' for t in tags))
print('-' * (22 + 16 * len(tags)))
for name, get in ROWS:
    cells = ''.join(f'{fmt(get(REPORTS[t]["metrics"])):>16s}' for t in tags)
    print(f'{name:22s}' + cells)

print()
print('Chênh lệch oracle - pipeline (dương = retrieval đang kéo xuống):')
for t in tags:
    m = REPORTS[t]['metrics']
    pipe = m['extraction']['normalized_arg_em_given_correct_tool']
    orac = (m['oracle_extraction'] or {}).get('normalized_arg_em_given_correct_tool')
    gap = None if (pipe is None or orac is None) else orac - pipe
    print(f'  {t:16s} pipeline={fmt(pipe)}  oracle={fmt(orac)}  chênh={fmt(gap)}')


metric                     custom_seen   custom_unseen       benchmark
----------------------------------------------------------------------
Call F1                         0.8869          0.8701          0.9890
Recall@5                        1.0000          1.0000          1.0000
Tool Set Accuracy               0.8550          0.7625          0.8028
ArgEM | đúng tool               0.6257          0.0984          0.4968
ArgEM | ORACLE tool             0.6425          0.1925          0.4858
Argument Pair F1                0.8513          0.5860          0.6151
Schema Validity                 0.8065          0.6799          0.8220
N-FCEM positive                 0.5350          0.0750          0.3984
Overall Success                 0.6400          0.4012          0.3984

Chênh lệch oracle - pipeline (dương = retrieval đang kéo xuống):
  custom_seen      pipeline=0.6257  oracle=0.6425  chênh=0.0168
  custom_unseen    pipeline=0.0984  oracle=0.1925  chênh=0.0941
  benchmark        pipeli

### Theo `tool_split` — Bảng D

`robustness` gom theo từng slice field. `metadata.tool_split` chỉ có ở
custom_vi (benchmark không gắn nhãn seen/unseen).


In [13]:
for t in tags:
    rob = REPORTS[t]['metrics'].get('robustness') or {}
    groups = (rob.get('metadata.tool_split') or {}).get('groups')
    if not groups:
        print(f'{t}: không có slice metadata.tool_split')
        continue
    print(f'--- {t} ---')
    hdr = f"{'split':10s} {'n':>6s} {'overall':>9s} {'N-FCEM':>9s} {'toolset':>9s} {'ArgEM':>9s}"
    print(hdr)
    for split, g in groups.items():
        print(f"{split:10s} {g['sample_count']:6d} {fmt(g['overall_success']):>9s}",
              f"{fmt(g['n_fcem_positive']):>9s} {fmt(g['tool_set_accuracy_positive']):>9s}",
              f"{fmt(g['normalized_arg_em_given_correct_tool']):>9s}")
    print()


--- custom_seen ---
split           n   overall    N-FCEM   toolset     ArgEM
none          400    0.7450      —         —         —   
seen          400    0.5350    0.5350    0.8550    0.6257

--- custom_unseen ---
split           n   overall    N-FCEM   toolset     ArgEM
none          400    0.7275      —         —         —   
test_unseen    400    0.0750    0.0750    0.7625    0.0984

benchmark: không có slice metadata.tool_split


## Kết quả — Bảng C và Bảng D (§11 `experimental_plan.md`)

Evaluator ghi `report.json` + `summary.md` + `per_sample.jsonl` cho mỗi
tập nhưng không in gì ra. Không đọc lại thì cả Phase 5 chỉ tạo file mà
không ai nhìn thấy con số nào.

**Chênh lệch ArgEM giữa pipeline và oracle** là số quan trọng nhất: nó
tách lỗi retrieval khỏi lỗi extraction (§6.3).


In [14]:
import collections

REPORTS = {}
for _, tag in GOLD:
    path = Path(f'results/evaluation/method_2_{tag}/report.json')
    if not path.exists():
        print(f'{tag}: THIẾU {path}')
        continue
    REPORTS[tag] = json.load(open(path, encoding='utf-8'))


def _f(value):
    return '     —' if value is None else f'{value:6.4f}'


hdr = (f"{'tập':15s} {'n':>6s} {'CallF1':>7s} {'ToolSet':>7s} {'ArgEM':>7s}"
       f" {'oracle':>7s} {'Δ':>7s} {'Schema':>7s} {'N-FCEM':>7s} {'Success':>7s}")
print(hdr)
print('-' * len(hdr))
for tag, rep in REPORTS.items():
    m = rep['metrics']
    arg = m['extraction']['normalized_arg_em_given_correct_tool']
    orc = (m['oracle_extraction'] or {}).get('normalized_arg_em_given_correct_tool')
    gap = None if (arg is None or orc is None) else orc - arg
    print(f"{tag:15s} {rep['dataset']['sample_count']:6d}",
          _f(m['detection']['f1']),
          _f(m['selection']['tool_set_accuracy_positive']),
          _f(arg), _f(orc), _f(gap),
          _f(m['schema_validity']['call_schema_validity']),
          _f(m['end_to_end']['n_fcem_positive']),
          _f(m['end_to_end']['overall_success']))

# Bảng D — seen vs unseen. `--slice metadata.tool_split` đi vào `robustness`.
print()
print('theo tool_split (Bảng D):')
for tag, rep in REPORTS.items():
    field = (rep['metrics'].get('robustness') or {}).get('metadata.tool_split')
    for value, g in ((field or {}).get('groups') or {}).items():
        print(f"  {tag:15s} {value:8s} n={g['sample_count']:5d}",
              'ToolSet=' + _f(g['tool_set_accuracy_positive']),
              'ArgEM=' + _f(g['normalized_arg_em_given_correct_tool']),
              'Success=' + _f(g['overall_success']))

# Phân loại lỗi W/T/P/I (§9). Đếm trên từng PARAMETER, không phải từng
# sample — nên tổng lớn hơn số sample là bình thường.
print()
print('phân loại lỗi (§9):')
for _, tag in GOLD:
    path = Path(f'results/method2/predictions/{tag}/errors_pipeline.jsonl')
    if not path.exists():
        continue
    counts = collections.Counter(
        json.loads(line)['error_class'] for line in open(path, encoding='utf-8')
    )
    print(f'  {tag:15s}', dict(counts.most_common()))


tập                  n  CallF1 ToolSet   ArgEM  oracle       Δ  Schema  N-FCEM Success
--------------------------------------------------------------------------------------
custom_seen        800 0.8869 0.8550 0.6257 0.6425 0.0168 0.8065 0.5350 0.6400
custom_unseen      800 0.8701 0.7625 0.0984 0.1925 0.0941 0.6799 0.0750 0.4012
benchmark        10555 0.9890 0.8028 0.4968 0.4858 -0.0110 0.8220 0.3984 0.3984

theo tool_split (Bảng D):
  custom_seen     none     n=  400 ToolSet=     — ArgEM=     — Success=0.7450
  custom_seen     seen     n=  400 ToolSet=0.8550 ArgEM=0.6257 Success=0.5350
  custom_unseen   none     n=  400 ToolSet=     — ArgEM=     — Success=0.7275
  custom_unseen   test_unseen n=  400 ToolSet=0.7625 ArgEM=0.0984 Success=0.0750

phân loại lỗi (§9):
  custom_seen     {'hallucinated_call': 190, 'T': 120, 'wrong_tool': 58, 'I': 55, 'P': 19, 'W': 4}
  custom_unseen   {'I': 266, 'hallucinated_call': 218, 'T': 129, 'W': 123, 'wrong_tool': 89, 'P': 47, 'missed_call': 9}
  benc

## Latency — tách 4 giai đoạn

§10.1 `experimental_plan.md` bắt buộc tách riêng thời gian embed query,
retrieve, cross-encode và validate. `t_index_build` là chi phí một lần,
báo cáo riêng, **không** cộng vào latency mỗi query.


In [15]:
for _, tag in GOLD:
    path = f'results/method2/predictions/{tag}/latency_pipeline.json'
    if not Path(path).exists():
        print(f'{tag}: chưa có {path}')
        continue
    latency = json.load(open(path, encoding='utf-8'))
    print(f'--- {tag} ---')
    for stage in ['t_query_embed', 't_retrieve', 't_cross_encode',
                  't_validate', 'total']:
        if stage in latency:
            print(f"  {stage:16s} p50={latency[stage]['p50_ms']:8.2f} ms",
                  f"p95={latency[stage]['p95_ms']:8.2f} ms")

print()
print('t_index_build (một lần, KHÔNG cộng vào latency/query):',
      meta['t_index_build_sec'], 's')


--- custom_seen ---
  t_query_embed    p50=   44.94 ms p95=   48.72 ms
  t_retrieve       p50=    0.08 ms p95=    0.12 ms
  t_cross_encode   p50=   13.74 ms p95=   41.25 ms
  t_validate       p50=    0.12 ms p95=    0.29 ms
  total            p50=   58.68 ms p95=   89.51 ms
--- custom_unseen ---
  t_query_embed    p50=   45.03 ms p95=   49.14 ms
  t_retrieve       p50=    0.08 ms p95=    0.11 ms
  t_cross_encode   p50=   13.31 ms p95=   47.12 ms
  t_validate       p50=    0.11 ms p95=    0.32 ms
  total            p50=   58.43 ms p95=   93.64 ms
--- benchmark ---
  t_query_embed    p50=   44.24 ms p95=   47.81 ms
  t_retrieve       p50=    0.06 ms p95=    0.09 ms
  t_cross_encode   p50=   11.98 ms p95=   33.47 ms
  t_validate       p50=    0.10 ms p95=    0.23 ms
  total            p50=   56.30 ms p95=   79.59 ms

t_index_build (một lần, KHÔNG cộng vào latency/query): 46.172 s


In [16]:
# ===== Lưu artifact =====
# Kaggle giữ /kaggle/working trong Output của version; tar chỉ là tiện lợi.
# `tar` báo lỗi nếu truyền đường dẫn không tồn tại, nên lọc trước — mỗi
# notebook sinh ra một tập thư mục khác nhau.
_want = ['artifacts/method2', 'results/method2', 'results/evaluation']
_have = [p for p in _want if (Path('/kaggle/working') / p).exists()]
print('đóng gói:', _have)
_paths = ' '.join(_have)
!tar czf /kaggle/working/eval_run.tar.gz -C /kaggle/working {_paths}
!du -h /kaggle/working/eval_run.tar.gz


đóng gói: ['artifacts/method2', 'results/method2', 'results/evaluation']
822M	/kaggle/working/eval_run.tar.gz
